# Pré-processamento de dados

In [24]:
# Importando as bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_aluno_original = pd.read_csv('dados/TS_ALUNO_34EM.csv', encoding='latin-1', sep=';')
df_diretor_original = pd.read_csv('dados/TS_DIRETOR.csv', encoding='latin-1', sep=';')
df_escola_original = pd.read_csv('dados/TS_ESCOLA.csv', encoding='latin-1', sep=';')

In [59]:
# ================================================================
# Escolhemos a features que temos interesses e as features
# que usaremos para limpar linhas que não tenham dados relevantes
# ================================================================

colunas_de_interesse_aluno = ['ID_ESCOLA', 'ID_ALUNO', 'ID_UF', 'ID_AREA', 'IN_PUBLICA', 'ID_SERIE', 'PROFICIENCIA_LP_SAEB', 
'PROFICIENCIA_MT_SAEB', 'TX_RESP_Q01', 'TX_RESP_Q02', 'TX_RESP_Q03', 'TX_RESP_Q04', 'TX_RESP_Q05a', 'TX_RESP_Q05b',
'TX_RESP_Q05c', 'TX_RESP_Q06','TX_RESP_Q07a','TX_RESP_Q07b','TX_RESP_Q07c','TX_RESP_Q07d','TX_RESP_Q07e','TX_RESP_Q08','TX_RESP_Q09',
'TX_RESP_Q10a','TX_RESP_Q10b','TX_RESP_Q10c','TX_RESP_Q10d','TX_RESP_Q10e','TX_RESP_Q10f','TX_RESP_Q11a','TX_RESP_Q11b','TX_RESP_Q11c',
'TX_RESP_Q12b','TX_RESP_Q12c','TX_RESP_Q14','TX_RESP_Q15b','TX_RESP_Q16','TX_RESP_Q17','TX_RESP_Q18','TX_RESP_Q19','TX_RESP_Q21a',
'TX_RESP_Q21b','TX_RESP_Q21c','TX_RESP_Q21d','TX_RESP_Q21e','TX_RESP_Q23d','TX_RESP_Q24']

remover_aluno = {
    'IN_PRESENCA_LP': 0,                    # Remover alunos que não tenham respondido a prova de língua portuguesa
    'IN_PRESENCA_MT': 0,                    # Remover alunos que não tenham respondido a prova de matemática
    'IN_PROFICIENCIA_LP': 0,                # Remover alunos que não tenham proficiência em língua portuguesa
    'IN_PROFICIENCIA_MT': 0,                # Remover alunos que não tenham proficiência em matemática
    'IN_PREENCHIMENTO_QUESTIONARIO': 0}     # Remover alunos que não preencheram o questionário

colunas_de_interesse_diretor = ['ID_ESCOLA','TX_Q020','TX_Q022','TX_Q032','TX_Q033','TX_Q035',
'TX_Q054','TX_Q056','TX_Q057','TX_Q078','TX_Q079','TX_Q081','TX_Q082','TX_Q083','TX_Q087',
'TX_Q108','TX_Q119','TX_Q129','TX_Q130','TX_Q139','TX_Q191','TX_Q194','TX_Q203','TX_Q207',
'TX_Q208','TX_Q209']

remover_diretor = {
    'IN_PREENCHIMENTO_QUESTIONARIO': [0]}     # Remover diretores que não preencheram o questionário

colunas_de_interesse_escola = ['ID_ESCOLA','PC_FORMACAO_DOCENTE_MEDIO','TAXA_PARTICIPACAO_EM',
    'MEDIA_EMT_LP','MEDIA_EMT_MT','MEDIA_EMI_LP','MEDIA_EMI_MT','MEDIA_EM_LP','MEDIA_EM_MT']

remover_escola = {
    'NU_PRESENTES_EM': 0}                   # Remover escolas que não tenham alunos presentes no ensino médio

In [61]:
# ================================================================
# Aplicaremos as regras de limpeza de dados e selecionaremos as features
# ================================================================

df_aluno_limpo = df_aluno_original.copy()
df_diretor_limpo = df_diretor_original.copy()
df_escola_limpo = df_escola_original.copy()

for coluna, valor in remover_aluno.items():
    antes = len(df_aluno_limpo)
    df_aluno_limpo = df_aluno_limpo[df_aluno_limpo[coluna] != valor]
    removidas = antes - len(df_aluno_limpo)

    print(
        f"Alunos: removidas {removidas} linhas "
        f"pela regra: {coluna} != {valor}"
    )

for coluna, valor in remover_diretor.items():
    antes = len(df_diretor_limpo)
    df_diretor_limpo = df_diretor_limpo[~df_diretor_limpo[coluna].isin(valor)]
    removidas = antes - len(df_diretor_limpo)

    print(
        f"Diretores: removidas {removidas} linhas "
        f"pela regra: {valor} não estar em {coluna}"
    )

for coluna, valor in remover_escola.items():
    antes = len(df_escola_limpo)
    df_escola_limpo = df_escola_limpo[df_escola_limpo[coluna] != valor]
    removidas = antes - len(df_escola_limpo)

    print(
        f"Escolas: removidas {removidas} linhas "
        f"pela regra: {coluna} != {valor}"
    )

df_aluno_limpo = df_aluno_limpo[colunas_de_interesse_aluno]
df_diretor_limpo = df_diretor_limpo[colunas_de_interesse_diretor]
df_escola_limpo = df_escola_limpo[colunas_de_interesse_escola]

print("\n# ================================================================\n")

print(f'Alunos: {len(df_aluno_limpo)} linhas')
print(f'Diretores: {len(df_diretor_limpo)} linhas')
print(f'Escolas: {len(df_escola_limpo)} linhas')

Alunos: removidas 562414 linhas pela regra: IN_PRESENCA_LP != 0
Alunos: removidas 0 linhas pela regra: IN_PRESENCA_MT != 0
Alunos: removidas 2966 linhas pela regra: IN_PROFICIENCIA_LP != 0
Alunos: removidas 0 linhas pela regra: IN_PROFICIENCIA_MT != 0
Alunos: removidas 11509 linhas pela regra: IN_PREENCHIMENTO_QUESTIONARIO != 0
Diretores: removidas 17304 linhas pela regra: [0] não estar em IN_PREENCHIMENTO_QUESTIONARIO
Escolas: removidas 122 linhas pela regra: NU_PRESENTES_EM != 0

# ================================================================

Alunos: 1514448 linhas
Diretores: 89785 linhas
Escolas: 70029 linhas


In [ ]:
# ================================================================
# Podemos ter problemas ao unificar as tabelas se houver escolas
# com o mesmo ID e multiplos diretores
# ================================================================

print(f"Alunos em mais de uma escola: {df_aluno_limpo['ID_ALUNO'].duplicated().sum()}")
print(f"Diretores com ID_ESCOLA duplicados: {df_diretor_limpo['ID_ESCOLA'].duplicated().sum()}")
print(f"Escolas com ID_ESCOLA duplicados: {df_escola_limpo['ID_ESCOLA'].duplicated().sum()}")

print("\n# ================================================================\n")

escolas_com_diretores_duplicados = (
    df_diretor_limpo['ID_ESCOLA']
    .value_counts()
    .loc[lambda x: x > 1]
)

escolas_sem_diretores = (
    df_escola_limpo['ID_ESCOLA']
    .loc[~df_escola_limpo['ID_ESCOLA'].isin(df_diretor_limpo['ID_ESCOLA'])]
)

print(f'Escolas com mais de um diretor: {len(escolas_com_diretores_duplicados)}')
print(f'Escolas sem diretor: {len(escolas_sem_diretores)}')

print("\n# ================================================================\n")

# ================================================================
# Vamos tratar os casos de escolas com mais de um diretor
# ================================================================

Alunos em mais de uma escola: 0
Diretores com ID_ESCOLA duplicados: 27915
Escolas com ID_ESCOLA duplicados: 0

# ================================================================

Escolas com mais de um diretor: 24853
Escolas sem diretor: 10253

# ================================================================



In [33]:
df_unificado = df_aluno_limpo.merge(df_diretor_limpo, on='ID_ESCOLA', how='inner').merge(df_escola_limpo, on='ID_ESCOLA', how='inner')

In [10]:
# Informações gerais sobre df_aluno
df_aluno.info()

print("================================================================================")
print("Forma do df_aluno:", df_aluno.shape)
print("================================================================================")
# Com o datset com */. para dados nulos ou brancos, é necessário substituir esses valores com NaN para facilitar a análise
df_aluno = df_aluno.replace(['.','*'], np.nan)
print("Quantidade de linhas com pelo menos um valor nulo:", df_aluno.isnull().any(axis=1).sum())

# Separação das colunas de interesse para a análise
colunasAluno_interesse = ['ID_ESCOLA', 'ID_ALUNO', 'ID_UF', 'ID_AREA', 'IN_PUBLICA', 'ID_SERIE', 'IN_PRESENCA_LP', 'IN_PRESENCA_MT', 'IN_PROFICIENCIA_LP', 'IN_PROFICIENCIA_MT', 'PROFICIENCIA_LP_SAEB',
'PROFICIENCIA_MT_SAEB', 'IN_PREENCHIMENTO_QUESTIONARIO','TX_RESP_Q01', 'TX_RESP_Q02', 'TX_RESP_Q03', 'TX_RESP_Q04', 'TX_RESP_Q05a', 'TX_RESP_Q05b', 'TX_RESP_Q05c', 'TX_RESP_Q06', 'TX_RESP_Q07a', 'TX_RESP_Q07b', 'TX_RESP_Q07c', 'TX_RESP_Q07d', 'TX_RESP_Q07e'
, 'TX_RESP_Q08', 'TX_RESP_Q09', 'TX_RESP_Q10a', 'TX_RESP_Q10b', 'TX_RESP_Q10c', 'TX_RESP_Q10d', 'TX_RESP_Q10e', 'TX_RESP_Q10f', 'TX_RESP_Q11a', 'TX_RESP_Q11b', 'TX_RESP_Q11c', 'TX_RESP_Q12b', 'TX_RESP_Q12c', 'TX_RESP_Q14', 'TX_RESP_Q15b'
, 'TX_RESP_Q16', 'TX_RESP_Q17', 'TX_RESP_Q18', 'TX_RESP_Q19', 'TX_RESP_Q21a', 'TX_RESP_Q21b', 'TX_RESP_Q21c', 'TX_RESP_Q21d', 'TX_RESP_Q21e', 'TX_RESP_Q23d', 'TX_RESP_Q24']

df_aluno = df_aluno[colunasAluno_interesse]
print("Formato para análise após a seleção das colunas de interesse:", df_aluno.shape)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2091337 entries, 0 to 2091336
Data columns (total 52 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   ID_ESCOLA                      int64  
 1   ID_ALUNO                       int64  
 2   ID_UF                          int64  
 3   ID_AREA                        int64  
 4   IN_PUBLICA                     int64  
 5   ID_SERIE                       int64  
 6   IN_PRESENCA_LP                 int64  
 7   IN_PRESENCA_MT                 int64  
 8   IN_PROFICIENCIA_LP             int64  
 9   IN_PROFICIENCIA_MT             int64  
 10  PROFICIENCIA_LP_SAEB           float64
 11  PROFICIENCIA_MT_SAEB           float64
 12  IN_PREENCHIMENTO_QUESTIONARIO  int64  
 13  TX_RESP_Q01                    object 
 14  TX_RESP_Q02                    object 
 15  TX_RESP_Q03                    object 
 16  TX_RESP_Q04                    object 
 17  TX_RESP_Q05a                   object 
 18  TX

In [4]:
# Visualização do dataset após a seleção das colunas de interesse
df_aluno.head()

,ID_ESCOLA,ID_ALUNO,ID_UF,ID_AREA,IN_PUBLICA,ID_SERIE,IN_PRESENCA_LP,IN_PRESENCA_MT,IN_PROFICIENCIA_LP,IN_PROFICIENCIA_MT,...,TX_RESP_Q17,TX_RESP_Q18,TX_RESP_Q19,TX_RESP_Q21a,TX_RESP_Q21b,TX_RESP_Q21c,TX_RESP_Q21d,TX_RESP_Q21e,TX_RESP_Q23d,TX_RESP_Q24
0,61412274,49408512,11,2,1,12,1,1,1,1,...,A,A,B,B,B,C,D,D,B,D
1,61412274,49408513,11,2,1,12,1,1,1,1,...,D,A,A,B,B,C,D,C,B,D
2,61412274,49408514,11,2,1,12,1,1,1,1,...,C,A,A,B,B,B,D,D,B,C
3,61412274,49408515,11,2,1,12,1,1,1,1,...,B,A,A,B,A,B,D,B,A,C
4,61412274,49408516,11,2,1,12,1,1,1,1,...,B,A,A,A,A,NaN,D,B,B,C


In [5]:
df_diretor.describe()

,ID_SAEB,ID_REGIAO,ID_UF,ID_MUNICIPIO,ID_AREA,ID_ESCOLA,IN_PUBLICA,ID_LOCALIZACAO,IN_PREENCHIMENTO_QUESTIONARIO,ID_SERIE,...,TX_Q020,TX_Q021,TX_Q022,TX_Q023,TX_Q024,TX_Q090,TX_Q111,TX_Q113,TX_Q114,TX_Q116
count,107089.0,107089.000000,107089.000000,1.070890e+05,107089.000000,1.070890e+05,107089.000000,107089.000000,107089.000000,107089.000000,...,88348.000000,88231.000000,88204.000000,84275.000000,87865.000000,25664.000000,78549.000000,74203.000000,73667.000000,46482.000000
mean,2023.0,2.765513,31.262819,6.324814e+06,1.855158,6.143512e+07,0.946082,1.220676,0.838415,7.822167,...,15.648515,15.490089,16.828817,12.906164,42.987788,10.731024,5.219837,3.595609,10.926168,5.004475
std,0.0,1.092463,10.061756,1.635145e+03,0.351943,2.127006e+04,0.225856,0.414705,0.368072,2.831646,...,12.796004,12.878979,13.600166,13.202094,8.712613,18.852395,2.952463,1.971914,14.082516,3.034731
min,2023.0,1.000000,11.000000,6.322170e+06,1.000000,6.139821e+07,0.000000,1.000000,0.000000,2.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2023.0,2.000000,24.000000,6.323322e+06,2.000000,6.141674e+07,1.000000,1.000000,1.000000,5.000000,...,8.000000,8.000000,8.000000,5.000000,40.000000,2.000000,3.000000,2.000000,0.000000,3.000000
50%,2023.0,3.000000,31.000000,6.324938e+06,2.000000,6.143519e+07,1.000000,1.000000,1.000000,9.000000,...,10.000000,10.000000,10.000000,8.000000,40.000000,5.000000,4.000000,3.000000,5.000000,4.000000
75%,2023.0,3.000000,35.000000,6.326028e+06,2.000000,6.145357e+07,1.000000,1.000000,1.000000,9.000000,...,20.000000,20.000000,20.000000,15.000000,46.000000,12.000000,7.000000,4.000000,16.000000,7.000000
max,2023.0,5.000000,53.000000,6.327738e+06,2.000000,6.147180e+07,1.000000,2.000000,1.000000,13.000000,...,60.000000,60.000000,60.000000,60.000000,60.000000,200.000000,12.000000,12.000000,50.000000,12.000000


In [6]:
import seaborn as sns
import matplotlib.pyplot as plt

# Configurações visuais do Seaborn
sns.set_theme(style="whitegrid")

# Apenas proficiências válidas
df_plot = df_aluno[(df_aluno['PROFICIENCIA_MT'] > 0) & (df_aluno['PROFICIENCIA_LP'] > 0)].copy()

# Mapeando o Sexo
if 'TX_RESP_Q01' in df_plot.columns:
    df_plot['Sexo'] = df_plot['TX_RESP_Q01'].map({'A': 'Masculino', 'B': 'Feminino'})

# Mapeando a Rede de Ensino
if 'IN_PUBLICA' in df_plot.columns:
    df_plot['Rede'] = df_plot['IN_PUBLICA'].map({1: 'Pública', 0: 'Privada'})


# Histograma de Matemática
plt.figure(figsize=(10, 5))
sns.histplot(df_plot['PROFICIENCIA_MT'], kde=True, color='skyblue', bins=30)
plt.title('Distribuição da Proficiência em Matemática')
plt.xlabel('Proficiência MT')
plt.ylabel('Frequência')
plt.show()

# Histograma de Língua Portuguesa
plt.figure(figsize=(10, 5))
sns.histplot(df_plot['PROFICIENCIA_LP'], kde=True, color='lightgreen', bins=30)
plt.title('Distribuição da Proficiência em Língua Portuguesa')
plt.xlabel('Proficiência LP')
plt.ylabel('Frequência')
plt.show()

if 'Sexo' in df_plot.columns:
    # Matemática por Sexo
    plt.figure(figsize=(8, 5))
    sns.boxplot(x='Sexo', y='PROFICIENCIA_MT', data=df_plot.dropna(subset=['Sexo']), palette='pastel')
    plt.title('Proficiência em Matemática por Sexo')
    plt.xlabel('Sexo')
    plt.ylabel('Proficiência Matemática')
    plt.show()
    
    # Língua Portuguesa por Sexo
    plt.figure(figsize=(8, 5))
    sns.boxplot(x='Sexo', y='PROFICIENCIA_LP', data=df_plot.dropna(subset=['Sexo']), palette='pastel')
    plt.title('Proficiência em Língua Portuguesa por Sexo')
    plt.xlabel('Sexo')
    plt.ylabel('Proficiência Língua Portuguesa')
    plt.show()

if 'Rede' in df_plot.columns:
    # Matemática por Rede
    plt.figure(figsize=(8, 5))
    sns.boxplot(x='Rede', y='PROFICIENCIA_MT', data=df_plot.dropna(subset=['Rede']), palette='Set2')
    plt.title('Proficiência em Matemática: Escola Pública vs. Privada')
    plt.xlabel('Rede de Ensino')
    plt.ylabel('Proficiência Matemática')
    plt.show()

    # Língua Portuguesa por Rede
    plt.figure(figsize=(8, 5))
    sns.boxplot(x='Rede', y='PROFICIENCIA_LP', data=df_plot.dropna(subset=['Rede']), palette='Set2')
    plt.title('Proficiência em Língua Portuguesa: Escola Pública vs. Privada')
    plt.xlabel('Rede de Ensino')
    plt.ylabel('Proficiência Língua Portuguesa')
    plt.show()


KeyError: 'PROFICIENCIA_MT'

# Análise Exploratória

# Modelos de Machine Learning

## Modelo 1

### Construção e treinamento

### Avaliação e métricas

## Modelo 2

### Construção e treinamento

### Avaliação e métricas

## Modelo 3

### Construção e treinamento

### Avaliação e métricas

## Modelo 4

### Construção e treinamento

### Avaliação e métricas

# Comparação dos Resultados